# Description

For this assignment, I chose to use the [Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents) multi-agent architecture. This architecture is ideal for situations where an agent needs to operate with multiple data sources and integrate their results.

The solution consists of a single coordinator agent and two subagents: one for interacting with the database and one for working with geospatial data.

### 1. Insert test data into the database and create a few utility functions

In [1]:
import psycopg
from contextlib import contextmanager

DB_NAME = "treesdb"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = 5432

@contextmanager
def open_db_cursor():
    conn = psycopg.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )

    yield conn.cursor()

    conn.commit()
    conn.close()

def get_create_table_statements() -> str:
    output = []
    with open_db_cursor() as cursor:
        cursor.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public' AND table_type = 'BASE TABLE'
        """)
        tables = cursor.fetchall()
            
        for (table_name,) in tables:
            cursor.execute(f"""
                SELECT 'CREATE TABLE ' || relname || E'\n(\n' ||
                array_to_string(
                    array_agg(
                        '    ' || column_name || ' ' || type || ' ' || not_null
                    ORDER BY ordinal_position
                    ), E',\n'
                ) || E'\n);\n'
                FROM (
                    SELECT c.relname, a.attname AS column_name,
                        pg_catalog.format_type(a.atttypid, a.atttypmod) AS type,
                        CASE WHEN a.attnotnull THEN 'NOT NULL' ELSE 'NULL' END AS not_null,
                        a.attnum AS ordinal_position
                    FROM pg_class c
                    JOIN pg_attribute a ON a.attrelid = c.oid
                    WHERE c.relname = '{table_name}'
                        AND a.attnum > 0
                        AND NOT a.attisdropped
                ) AS cols
                GROUP BY relname
            """)
            result = cursor.fetchone()
            if result:
                output.append(result[0])

            # Extract CHECK constraints
            cursor.execute(f"""
                SELECT con.conname, pg_get_constraintdef(con.oid)
                FROM pg_constraint con
                JOIN pg_class rel ON rel.oid = con.conrelid
                JOIN pg_namespace nsp ON nsp.oid = rel.relnamespace
                WHERE rel.relname = '{table_name}'
                    AND nsp.nspname = 'public'
                    AND con.contype IN ('c', 'f', 'p', 'u')
            """)
            constraints = cursor.fetchall()
            if constraints:
                output.append(f"-- Constraints for {table_name}:")
                for (con_name, con_def) in constraints:
                    output.append(f"    CONSTRAINT {con_name} {con_def}")
                output.append("")

    return "\n".join(output)

    
sql_file = "data/db.sql"
with open(sql_file, 'r') as f:
    sql_script = f.read()

with open_db_cursor() as cursor:
    try:
        cursor.execute(sql_script)
        print("SQL script executed successfully.")
    except Exception as e:
        print("Error executing SQL script:", e)

SQL script executed successfully.


### 2. Add utility functions for LangChain

In [2]:
import os

from langchain_anthropic import ChatAnthropic
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

_PRINT_TOOL_ARGS=False

def mk_agent(tools: list[callable], system_prompt: str, use_checkpoint: bool = False):
    model = ChatAnthropic(
        model="claude-sonnet-4-5",
        max_tokens=10000,
        api_key=os.getenv('ANTHROPIC_API_KEY')
    )
    checkpointer = InMemorySaver() if use_checkpoint else None
    agent = create_agent(
        model=model,
        system_prompt=system_prompt,
        tools=tools,
        checkpointer=checkpointer
    )
    return agent


def invoke_agent(agent, prompt: str, thread_id: str = '1') -> str:
    response = agent.invoke(
        {"messages": [
            {"role": "user", "content": prompt}
        ]}, 
        {"configurable": {"thread_id": thread_id}}
    )
    return response['messages'][-1].content

### 3. Create Aget for interaction with DB 

In [ ]:
from langchain.tools import tool

@tool
def get_db_schema() -> str:
    '''
    Returns the schema of the Postgresql database.
    '''
    return get_create_table_statements()

@tool
def execute_sql_query(query: str) -> str:
    '''
    Executes a SQL query against the Postgresql database and returns the results.
    '''
    if _PRINT_TOOL_ARGS:
        print(f"\033[33mexecute_sql_query(query={query})\033[0m")
    with open_db_cursor() as cursor:
        cursor.execute(query)
        results = cursor.fetchall()
        return str(results)


DB_AGENT_PROMPT = """
You are a helpful assistant for working with a Postgresql database. You have access to the following tools:
get_db_schema: Returns the schema of the Postgresql database.        
execute_sql_query: Executes a SQL query against the Postgresql database and returns the results.

Try to be as efficient as possible with your queries, and only query for the data you need to answer the user's question.
Produce as short answer as possible, and only include relevant information.
Do not modify the database in any way, only read data from it.
"""

db_agent = mk_agent(
    tools=[get_db_schema, execute_sql_query],
    system_prompt=DB_AGENT_PROMPT
)

test_db_prompts = [
    "How many trees are healthy in `North Field`?",
]

for prompt in test_db_prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {invoke_agent(db_agent, prompt, thread_id='db_test_thread_1')}")
    print()

Prompt: How many trees are healthy in `North Field`?
Response: There are **6 healthy trees** in North Field.



### 4. Add an agent for quering the geospital data

In [4]:
from functools import cache
import geopandas as gpd
from shapely.geometry import Point
from langchain.tools import tool


@cache
def load_water_polygons(file_path: str) -> gpd.GeoDataFrame:
    """Load water polygons from a GeoJSON file."""
    gdf = gpd.read_file(file_path)
    gdf['date'] = gpd.pd.to_datetime(gdf['date'])
    return gdf


@tool
def find_water_bodies_containing_point(coordinates: list[tuple[float, float]], field_ids:list[int], start_date: str = None, end_date: str = None) -> str:
    """
    Find water bodies containing the given latitude and longitude.
    Args:
        coordinates: A list of tuples, where each tuple contains the latitude and longitude of a point. 
        If it is empty, it should return all water bodies for the given dates and field_ids.
        start_date: The start date to filter the water bodies by (in YYYY-MM-DD format). If it is empty, it should return all water bodies for the given coordinates and field_ids.
        end_date: The end date to filter the water bodies by (in YYYY-MM-DD format). If it is empty, it should return all water bodies for the given coordinates and field_ids.
        field_ids: A list of field IDs to filter the water bodies by. if it is empty, it should return all water bodies for the given coordinates and dates.
    """
    if _PRINT_TOOL_ARGS:
        print(f"\033[32mfind_water_bodies_containing_point(coordinates={coordinates}, field_ids={field_ids}, start_date={start_date}, end_date={end_date})\033[0m")
    
    polygons = load_water_polygons("data/water_polygons.geojson")

    if coordinates:
        points = [Point(lon, lat) for lon, lat in coordinates]
        points_gdf = gpd.GeoDataFrame(geometry=points, crs=polygons.crs)
        polygons = gpd.sjoin(points_gdf, polygons, predicate='within')

    if start_date or end_date:
        if start_date:
            start_date = gpd.pd.to_datetime(start_date)
            polygons = polygons[polygons['date'] >= start_date]
        if end_date:
            end_date = gpd.pd.to_datetime(end_date)
            polygons = polygons[polygons['date'] <= end_date]
    
    if field_ids:
        polygons = polygons[polygons['field_id'].isin(field_ids)]

    polygons = polygons[['field_id', 'date']]    
    return str(polygons) if not polygons.empty else "No water bodies found containing the given points."

    

GEO_AGENT_PROMPT = """
You are a helpful assistant for working with a geospatial dataset of water bodies. You have access to the following tool:
find_water_bodies_containing_point: Find water bodies containing the given latitude and longitude, and filter them by date and field_id if provided.
The dataset schema is as follows:
- field_id: string
- date: date
- geometry: polygon
"""

geo_agent = mk_agent(
    tools=[find_water_bodies_containing_point],
    system_prompt=GEO_AGENT_PROMPT
)

geo_test_promts = [
    'Was there any water at the point (1.5, 2.5) between 2026-04-01 and 2026-04-10?',
]

for prompt in geo_test_promts:
    print(f"Prompt: {prompt}")
    print(f"Response: {invoke_agent(geo_agent, prompt, thread_id='geo_test_thread_1')}")
    print()

Prompt: Was there any water at the point (1.5, 2.5) between 2026-04-01 and 2026-04-10?
Response: Yes, there was water at the point (1.5, 2.5) between 2026-04-01 and 2026-04-10. Specifically, water bodies were detected at that location on:

1. **2026-04-03** (field_id: 1)
2. **2026-04-05** (field_id: 2)

So water was present at that point on at least these two dates during the specified time period.



### 5. Create the coordinator agnet

In [5]:
@tool
def invoke_db_agent_tool(prompt: str) -> str:
    '''Invokes the database agent with the given prompt and returns the response.
    Args:
        prompt: The prompt to send to the database agent.
    Returns:
        The response from the database agent.
    ''' 
    if _PRINT_TOOL_ARGS:
        print(f"\033[34minvoke_db_agent_tool(prompt={prompt})\033[0m")
    return invoke_agent(db_agent, prompt)

@tool
def invoke_geo_agent_tool(prompt: str) -> str:
    '''Invokes the geospatial agent with the given prompt and returns the response.
    Args:
        prompt: The prompt to send to the geospatial agent.
    Returns:
        The response from the geospatial agent.
    '''
    if _PRINT_TOOL_ARGS:
        print(f"\033[34minvoke_geo_agent_tool(prompt={prompt})\033[0m")
    return invoke_agent(geo_agent, prompt)

COORDINATOR_PROMPT = """
You are a helpful assistant for answering questions about a farm. You have access to the following tools:
invoke_db_agent: Invokes the database agent with a given prompt and returns the response.
invoke_geo_agent: Invokes the geospatial agent with a given prompt and returns the response.
Use db agent when you need to get information about the state of trees on the give location or field at the given time,
 and use geo agent when you need to get information about water bodies (coverage) on the farm.   
"""

coordinator_agent = mk_agent(
    tools=[invoke_db_agent_tool, invoke_geo_agent_tool],
    system_prompt=COORDINATOR_PROMPT,
    use_checkpoint=True
)


### 6. Invoke Coordinator agent with test promts

In [6]:
coordinator_test_prompts = [
    'How many trees are healthy in East Orchard?',
    'Which trees were covered by water and became stressed?',
    'Count trees with a specific crop type in South Grove',
    'Where are dead citrus trees?',
]

for i, prompt in enumerate(coordinator_test_prompts):
    print(f"Prompt {i+1}: {prompt}")
    print(f"Response: {invoke_agent(coordinator_agent, prompt, thread_id=f'coordinator_thread_{i+111}')}")
    print('\n--------\n')

Prompt 1: How many trees are healthy in East Orchard?
Response: There are **5 healthy trees** in East Orchard.

--------

Prompt 2: Which trees were covered by water and became stressed?
Response: Based on my analysis, **3 trees were covered by water and became stressed**:

1. **Tree ID 2 (Apple)** in Field 1 at coordinates (1.5, 2.5)
   - Covered by water on field 1 on April 3, 2026 and field 2 on April 5, 2026

2. **Tree ID 19 (Apple)** in Field 2 at coordinates (3.5, 3.5)
   - Covered by water on field 3 on April 10, 2026

3. **Tree ID 25 (Orange)** in Field 3 at coordinates (1.5, 1.5)
   - Covered by water on field 3 on April 3, 2026

The other 3 stressed trees (IDs 7, 14, and 30) became stressed for reasons other than water coverage.

--------

Prompt 3: Count trees with a specific crop type in South Grove
Response: I'd be happy to help you count trees with a specific crop type in South Grove. However, I need to know which crop type you're interested in.

Could you please tell me 

In [7]:
THREAD_PROMPTS = [
    'Find fields covered with water on Wednesday.',
    'Just any Wednesday in April 2026.',
]

for prompt in THREAD_PROMPTS:
    print(f"Prompt: {prompt}")
    print(f"Response: {invoke_agent(coordinator_agent, prompt, thread_id='coordinator_thread_100')}")
    print('--------')


Prompt: Find fields covered with water on Wednesday.
Response: The geospatial agent needs more specific information to help you. To find fields covered with water on Wednesday, please provide:

1. **Which Wednesday?** - Please specify the date in YYYY-MM-DD format (for example: 2024-01-10)
   - Or you can specify which week/month/year you're interested in

2. **Location (optional)** - Do you want to search:
   - All available fields on the farm?
   - Specific field IDs?
   - A particular area with coordinates?

Once you provide the date information, I'll be able to find the fields that were covered with water on that specific Wednesday.
--------
Prompt: Just any Wednesday in April 2026.
Response: Based on the search results, here's what I found:

**Fields covered with water on Wednesdays in April 2026:**

- **Field 1** was covered with water on:
  - Wednesday, April 8, 2026
  - Wednesday, April 15, 2026

No water coverage was detected on the other Wednesdays in April 2026 (April 1st, 2